# [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cene555/ruCLIP-SB/blob/main/notebooks/evaluate_ruCLIP_SB_latest.ipynb)

In [3]:
!pip install open_clip_torch

In [1]:
import torch, open_clip
from PIL import Image
from pathlib import Path

# 1. Загружаем модель и препроцессоры
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = 'hf-hub:laion/CLIP-ViT-L-14-laion2B-s32B-b82K'
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(model_name, device=device)
tokenizer = open_clip.get_tokenizer(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
# 2. Готовим текстовые промпты «рукописный» и «печатный»
prompts = ["handwritten text", "printed text"]
with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):
    text_tokens   = tokenizer(prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)   # L2-норма

<ipython-input-2-64d0f0a31458>:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):


In [3]:
# 3. Функция для разовой классификации изображения
def classify(path: str | Path):
    img = Image.open(path).convert("RGB")
    img_tensor = preprocess_val(img).unsqueeze(0).to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):
        img_feat = model.encode_image(img_tensor)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        logits   = 100.0 * img_feat @ text_features.T     # scale, как в CLIP
        probs    = logits.softmax(dim=-1).squeeze(0)

    label = "handwritten" if probs.argmax().item() == 0 else "printed"
    return label, probs.tolist()

In [7]:
# 4. Пример запуска
print(classify("1.jpg"))
print(classify("2.jpg"))

('handwritten', [0.9998979568481445, 0.00010204206046182662])
('handwritten', [0.8736108541488647, 0.12638913094997406])


<ipython-input-3-a415993b8b7a>:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):


In [5]:
!ls

1.jpg  2.jpg  sample_data
